# L54 · Value 计算图（computation graph） — 标量自动微分（automatic differentiation，autograd）：前向值 + 反向梯度（gradient），手写 add / mul 节点

**目标**：实现计算图的最小单元 `Value`——存储前向值与梯度，记录生成它的算子（operator），为反向传播（backpropagation）打地基。

🔗 **Aurora 连接**：`Value` 是 Aurora ML 引擎（`src/aurora/ml/`，将在 Month 2 建立）的起点；Month 2 的所有训练循环（线性层、损失函数（loss function）、优化器）都以此节点为基础构建计算图，再调用 `.backward()` 自动求导。

← **上一课**　[L53 · MFCC 图形化](../5_audio_dsp/L53_visual_mfcc.ipynb)

> 上节课学习了 **MFCC 图形化**：波形 → 声谱图 → Mel 谱 → 倒谱系数，逐层图示。  
> 本课将探讨 **Value 计算图**。

## 本课剧情：PyTorch 的核心魔法是什么？

你写了 `loss.backward()`，PyTorch 自动算出每个参数的梯度。背后是什么？

**微积分链式法则（chain rule）**：如果 `z = f(g(x))`，则 `dz/dx = dz/dg · dg/dx`。神经网络就是一大串函数复合，层层套用链式法则，就能从输出的梯度"逆向"推导出每个参数的梯度——这叫**反向传播（backpropagation）**。

**问题**：损失函数 `L = tanh(a*b + b²)` 有 2 个参数，手写偏导数需要：
```
∂L/∂a = (1 - tanh²(a*b+b²)) · b
∂L/∂b = (1 - tanh²(a*b+b²)) · (a + 2b)
```
这是 2 个参数的情况。神经网络有数百万参数，不可能逐一手写。

**Karpathy micrograd 的洞察**：Python 对象图 = 计算图。每次写 `c = a + b`，就隐式建了一个 `+` 节点：

```
a ──┐
    ├──[+]──► c      c.data = a.data + b.data
b ──┘         c._backward() → a.grad += 1*c.grad, b.grad += 1*c.grad
```

反向传播 = 从输出节点开始，沿有向无环图（DAG）逆向遍历，把梯度从后往前一层层传递。

本节任务：实现最小化 `Value` 类，支持 `__add__`、`__mul__`、`__pow__` 和 `backward()`。

In [ ]:
# 本课纯 Python 实现，无需 numpy
# （本课不使用任何 numpy 功能——Value 类只用 Python 内建类型）

## 1. `Value` 的三个核心字段

每个 `Value` 节点携带三件事：

- `data`：前向计算的标量值，例如 `3.14`
- `grad`：该节点对最终损失 L 的偏导 `dL/d(self)`，**初始化为 0.0**（累积语义，不覆盖）
- `_backward`：一个**闭包（closure）**，调用时把当前节点的 `grad` 按链式法则传播给它的输入节点

`grad` 初始化为 `0.0` 而非 `None`，是因为同一个节点可能被多条路径依赖（如 `b` 在 `a*b + b**2` 中出现两次），梯度需要**累加**而非覆盖。

In [ ]:
# 演示：grad 累积的必要性
# 若 b 被用了两次，两条路径的梯度必须相加
# 这里只是数值演示，Value 类在下方实现
b_grad_from_path1 = 3.0   # 来自 a*b 对 b 的偏导
b_grad_from_path2 = 6.0   # 来自 b**2 对 b 的偏导（b=3 时 2*3=6）
total = b_grad_from_path1 + b_grad_from_path2
print(f"b.grad 应累积为 {total}（不是覆盖为 {b_grad_from_path2}）")

## 2. 计算图是 DAG

每次写 `c = a + b`，都隐式创建了一张**有向无环图（DAG）**：

```
a ──┐
    ├─[+]─► c
b ──┘
```

- **节点**：每个 `Value` 对象
- **边**：算子（`+`、`*`、`**` 等）；边的方向是数据流方向（前向）
- **`_prev`**：每个节点存储它的直接输入节点集合（`{a, b}`），反向时靠这张图回溯

DAG 无环是关键——如果有环，梯度会无限循环。在前馈网络里，数据从输入到损失单向流动，自然是 DAG。

In [ ]:
# 用字典模拟 DAG 结构，演示拓扑关系
# （真实 Value 类在下方；此处仅展示图结构思路）
graph = {
    "c": {"op": "+", "inputs": ["a", "b"]},
    "d": {"op": "*", "inputs": ["c", "b"]},
}
def topo(node, graph, visited=None, order=None):
    if visited is None: visited, order = set(), []
    if node not in visited:
        visited.add(node)
        for inp in graph.get(node, {}).get("inputs", []):
            topo(inp, graph, visited, order)
        order.append(node)
    return order

print("拓扑序（前向）:", topo("d", graph))
print("反向传播顺序:", list(reversed(topo("d", graph))))

## 3. 为什么用 Python 类

Karpathy micrograd 的核心洞察：**Python 对象图 = 计算图**。

不需要专门的图数据库，只需重载 `__add__`、`__mul__` 等运算符——每次运算自动创建新节点并记录父子关系。Python 的垃圾回收负责内存，`_prev` 集合维护图结构，`_backward` 闭包捕获所有需要的局部变量。

对比手动求导：假设 `L = x * y + y**2`，手写 `dL/dx = y`、`dL/dy = x + 2y`。当网络有百万参数时，手写不可行——`Value` 让每个算子只需知道自己的**一阶局部偏导**，反向传播自动组合它们。

In [ ]:
# 运算符重载的语法糖演示（伪代码对比）
print("手动求导（不可扩展）：")
print("  L = x*y + y**2")
print("  dL/dx = y")
print("  dL/dy = x + 2*y")
print()
print("Value 方案：每个算子只写一次局部偏导，链式法则自动组合")
print("  __mul__ 的 _backward: a.grad += b.data * out.grad")
print("                         b.grad += a.data * out.grad")

## 4. ✏️ 实现 `class Value`

**三个核心字段**：

| 字段 | 含义 | 初始值 |
|---|---|---|
| `data` | 标量前向值 | 构造时传入 |
| `grad` | 对最终损失的梯度 | 0.0（等待反向传播填写） |
| `_backward` | 该算子的反向函数 | `lambda: None` |

**四步实现路线**：

| 步骤 | 方法 | backward 公式 |
|---|---|---|
| 1 | `__init__` | — |
| 2 | `__add__` | `a.grad += out.grad`，`b.grad += out.grad` |
| 3 | `__mul__` | `a.grad += b.data * out.grad`，`b.grad += a.data * out.grad` |
| 4 | `backward()` | 拓扑排序 → 逆序调用每个节点的 `_backward()` |

**验收标准**：
- `(Value(2) + Value(3)).data == 5`
- `L = a*b + b**2`，`L.backward()`，`a.grad == b.data`（对a的偏导=b）

In [ ]:
class Value:
    def __init__(self, data, _children=(), _op=""):
        # ✏️ TODO: 存 data（转 float）、grad=0.0、_backward=lambda:None
        #          _prev=set(_children)、_op=_op
        raise NotImplementedError("请实现 __init__：设置 self.data, self.grad, self._backward, self._prev, self._op")

    def __repr__(self):
        # 使用 getattr 防止在桩状态（__init__ 未实现时）引发 AttributeError
        return f"Value(data={getattr(self, 'data', '?')}, grad={getattr(self, 'grad', '?')})"

    def __add__(self, other):
        # ✏️ TODO: other 若不是 Value 则包装；创建 out；定义 _backward（加法偏导均为1）
        raise NotImplementedError("请实现 __add__：包装 other，创建 out，设置 _backward")

    def __radd__(self, other): return self + other

    def __mul__(self, other):
        # ✏️ TODO: 乘积法则 _backward
        raise NotImplementedError("请实现 __mul__：包装 other，创建 out，用乘积法则设置 _backward")

    def __rmul__(self, other): return self * other

    def __pow__(self, other):
        # ✏️ TODO: 幂函数偏导：other * self.data**(other-1)
        assert isinstance(other, (int, float)), "指数必须是标量"
        raise NotImplementedError("请实现 __pow__：创建 out，设置 _backward（偏导 = other * self.data**(other-1)）")

    def __neg__(self):  return self * -1
    def __sub__(self, other): return self + (-other)
    def __rsub__(self, other): return other + (-self)
    def __truediv__(self, other): return self * other**-1

    def backward(self):
        # ✏️ TODO: 拓扑排序整图，逆序调用 _backward；先设 self.grad=1.0
        # 提示：参考 Cell 6 的 topo() 函数，用 DFS + visited 集合收集后序节点，再反转
        raise NotImplementedError("请实现 backward：先 self.grad=1.0，拓扑排序后逆序调用各节点 _backward")

In [ ]:
# 检查 __add__ 和 __mul__ 的前向值
a = Value(2.0); b = Value(3.0)
c = a + b
assert c.data == 5.0, f"期望 5.0，得到 {c.data}"
print("✅ a + b 前向正确：", c)

d = a * b
assert d.data == 6.0, f"期望 6.0，得到 {d.data}"
print("✅ a * b 前向正确：", d)

# 检查 __mul__ 的反向（不调用 backward，手动设 grad）
a2 = Value(2.0); b2 = Value(3.0)
out = a2 * b2
out.grad = 1.0
out._backward()
assert a2.grad == 3.0, f"期望 a.grad=3.0，得到 {a2.grad}"
assert b2.grad == 2.0, f"期望 b.grad=2.0，得到 {b2.grad}"
print("✅ __mul__ _backward 正确：a.grad=", a2.grad, " b.grad=", b2.grad)

# 检查完整 backward()
a3 = Value(4.0)
b3 = Value(2.0)
L = a3 * b3 + b3 ** 2   # L = 4*2 + 4 = 12；dL/da=2, dL/db=4+4=8
L.backward()
assert a3.grad == 2.0, f"期望 dL/da=2.0，得到 {a3.grad}"
assert b3.grad == 8.0, f"期望 dL/db=8.0，得到 {b3.grad}"
print("✅ L=(a*b + b**2) backward() 正确：a.grad=", a3.grad, " b.grad=", b3.grad)

## 5. 参数实验：手算 vs 自动微分

构建 `L = (a * b) + b**2`，取 `a=3.0, b=2.0`。

**手算**：
- `L = 3*2 + 2**2 = 10`
- `dL/da = b = 2`
- `dL/db = a + 2*b = 3 + 4 = 7`

**预期现象**：调用 `L.backward()` 后，`a.grad==2.0`，`b.grad==7.0`。

改变 `a` 和 `b` 的值，观察梯度如何随输入线性/非线性变化。

In [ ]:
# 参数实验：改变 a, b 的值，验证梯度公式
for a_val, b_val in [(3.0, 2.0), (1.0, 5.0), (-2.0, 3.0)]:
    a = Value(a_val); b = Value(b_val)
    L = a * b + b ** 2
    L.backward()
    # 手算期望
    expected_da = b_val
    expected_db = a_val + 2 * b_val
    print(f"a={a_val}, b={b_val} → L={L.data:.1f} | "
          f"a.grad={a.grad:.1f}（期望{expected_da:.1f}） "
          f"b.grad={b.grad:.1f}（期望{expected_db:.1f}）")
    assert abs(a.grad - expected_da) < 1e-9
    assert abs(b.grad - expected_db) < 1e-9
print("✅ 全部参数组合验证通过")

## 本课收束

`Value` 类通过 `__add__`、`__mul__`、`__pow__` 三个运算符重载构建计算图，每个算子在 `_backward` 闭包里写入自己的局部偏导，`backward()` 按拓扑逆序触发所有 `_backward`，最终在每个叶节点的 `.grad` 里得到 `dL/d(leaf)`。这套机制输出的是**标量梯度**，对应 Aurora ML 引擎的核心概念（`src/aurora/ml/` 为计划中模块，尚未建立；当前自动微分实现直接在 notebook 中演示）。下一节（**L55**）将补全 `__pow__`、`tanh`、`relu`、`exp` 算子，并在 `Value` 上搭建 `Neuron → Layer → MLP`，用同一套图机制训练一个能学习的小网络。

## ✏️ 闭卷推导检查格 — 计算图反向传播

**规则：关闭上方所有格，仅凭记忆完成以下推导。**

**题目**：给定表达式 $z = x \cdot y + y^2$，其中 $x = 3, y = 2$：

1. 画出前向计算图（标注所有中间节点的值）
2. 手写出链式展开，求 $\partial z / \partial x$ 和 $\partial z / \partial y$
3. 验证你的答案：$\partial z / \partial x = ?$，$\partial z / \partial y = ?$

（在此处写推导...）

In [ ]:
# 验证：用 aurora Value 类与手算梯度对比
import sys; sys.path.insert(0, 'src')
try:
    from aurora.llm.autograd import Value
    x = Value(3.0); y = Value(2.0)
    z = x * y + y ** 2
    z.backward()
    dz_dx = x.grad
    dz_dy = y.grad
except ImportError:
    # fallback: PyTorch
    import torch
    x = torch.tensor(3.0, requires_grad=True)
    y = torch.tensor(2.0, requires_grad=True)
    z = x * y + y ** 2
    z.backward()
    dz_dx, dz_dy = x.grad.item(), y.grad.item()

# z = xy + y^2 → dz/dx = y = 2, dz/dy = x + 2y = 3 + 4 = 7
assert abs(dz_dx - 2.0) < 1e-6, f"dz/dx = {dz_dx}，期望 2"
assert abs(dz_dy - 7.0) < 1e-6, f"dz/dy = {dz_dy}，期望 7"
print(f"✅ dz/dx = {dz_dx} ✓   dz/dy = {dz_dy} ✓")

In [ ]:
# ✏️ 本课自评
l54_review = {
    "chain_rule_understood":   None,  # 理解链式法则：dz/dx=dz/dg·dg/dx？True/False
    "value_class_impl":        None,  # Value.__add__/__mul__/__pow__/backward 全实现？True/False
    "backprop_dag_order":      None,  # 理解反向传播=拓扑逆序调用_backward？True/False
    "grad_accumulation":       None,  # 理解同一节点被多次使用时 grad 需要累加？True/False
    "whiteboard_passed":       None,  # 白板挑战计算图反向传播推导通过？True/False
}

unfilled = [k for k, v in l54_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l54_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L54 全部通关！进入 L55：Value 算子补全')

---

→ **下一课**　[L55 · Value 算子补全](L55_forward_pass.ipynb)

> 下节课将学习 **Value 算子补全**：pow、relu、tanh、exp 节点实现，计算图完整展开。